In [2]:
!pip install datasets opencv-python deepface scikit-learn -q

from datasets import load_dataset
import numpy as np
import cv2
from collections import Counter

print("CelebA yükleniyor...")
dataset = load_dataset("eurecom-ds/celeba", split="train")

# Sadece identity kolonunu çek (hızlı)
print("Kimlikler taranıyor...")
all_ids = dataset["identity"]
counter = Counter(all_ids)
top100 = set([k for k, _ in counter.most_common(100)])
print(f"Top 100 kişi belirlendi, en fazla fotoğraf: {counter.most_common(1)[0][1]}")

# Sadece o kişilerin indexlerini bul
indices = [i for i, uid in enumerate(all_ids) if uid in top100]
subset = dataset.select(indices)
print(f"Seçilen fotoğraf sayısı: {len(subset)}")

print("Veriler işleniyor...")
X_raw, y_raw, rgb_imgs = [], [], []
for item in subset:
    img = np.array(item["image"])
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray_resized = cv2.resize(gray, (47, 62))
    X_raw.append(gray_resized.flatten())
    y_raw.append(item["identity"])
    rgb_imgs.append(cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

X = np.array(X_raw)
y = np.array(y_raw)

print(f"Toplam fotoğraf: {X.shape[0]}")
print(f"Toplam kişi: {len(np.unique(y))}")
print("Veri hazır!")

CelebA yükleniyor...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/18 [00:00<?, ?it/s]

Kimlikler taranıyor...
Top 100 kişi belirlendi, en fazla fotoğraf: 35
Seçilen fotoğraf sayısı: 3024
Veriler işleniyor...
Toplam fotoğraf: 3024
Toplam kişi: 100
Veri hazır!


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("1. Eğitim/Test ayrımı yapılıyor...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("2. PCA eğitiliyor...")
pca = PCA(n_components=150, svd_solver='randomized', whiten=True).fit(X_train)
X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

print("3. Sınıflandırıcı eğitiliyor...")
clf = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_train_pca, y_train)

print("4. Test ediliyor...")
y_pred = clf.predict(X_test_pca)
y_prob = clf.predict_proba(X_test_pca)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
FP = cm.sum(axis=0) - np.diag(cm)
FN = cm.sum(axis=1) - np.diag(cm)
TP = np.diag(cm)
TN = cm.sum() - (FP + FN + TP)

FAR = np.mean(FP / (FP + TN + 1e-9))
FRR = np.mean(FN / (FN + TP + 1e-9))
EER = (FAR + FRR) / 2

classes = clf.classes_
y_test_bin = label_binarize(y_test, classes=classes)
auc_score = np.mean([
    auc(*roc_curve(y_test_bin[:, i], y_prob[:, i])[:2])
    for i in range(len(classes))
    if y_test_bin[:, i].sum() > 0
])

print("\n" + "="*40)
print("--- PCA (EIGENFACES) SONUÇLARI ---")
print("="*40)
print(f"Accuracy (Doğruluk): %{acc*100:.2f}")
print(f"FAR (Yanlış Kabul):  %{FAR*100:.2f}")
print(f"FRR (Yanlış Red):    %{FRR*100:.2f}")
print(f"EER (Eşit Hata):     %{EER*100:.2f}")
print(f"AUC:                  {auc_score:.4f}")
print("="*40)

1. Eğitim/Test ayrımı yapılıyor...
2. PCA eğitiliyor...
3. Sınıflandırıcı eğitiliyor...
4. Test ediliyor...

--- PCA (EIGENFACES) SONUÇLARI ---
Accuracy (Doğruluk): %32.89
FAR (Yanlış Kabul):  %0.68
FRR (Yanlış Red):    %67.19
EER (Eşit Hata):     %33.93
AUC:                  0.8856


In [4]:
from deepface import DeepFace
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("FaceNet embedding çıkarılıyor (2-3 dakika sürebilir)...")
embeddings = []
for i, img in enumerate(rgb_imgs):
    res = DeepFace.represent(img, model_name="Facenet", enforce_detection=False)
    embeddings.append(res[0]["embedding"])
    if i % 50 == 0:
        print(f"  {i}/{len(rgb_imgs)} tamamlandı...")

embeddings = np.array(embeddings)

X_tr, X_te, y_tr, y_te = train_test_split(
    embeddings, y, test_size=0.2, random_state=42, stratify=y
)

clf_fn = SVC(kernel='linear', class_weight='balanced', probability=True).fit(X_tr, y_tr)
y_pred_fn = clf_fn.predict(X_te)
y_prob_fn = clf_fn.predict_proba(X_te)

acc_fn = accuracy_score(y_te, y_pred_fn)
cm_fn = confusion_matrix(y_te, y_pred_fn)
FP2 = cm_fn.sum(axis=0) - np.diag(cm_fn)
FN2 = cm_fn.sum(axis=1) - np.diag(cm_fn)
TP2 = np.diag(cm_fn)
TN2 = cm_fn.sum() - (FP2 + FN2 + TP2)

FAR2 = np.mean(FP2 / (FP2 + TN2 + 1e-9))
FRR2 = np.mean(FN2 / (FN2 + TP2 + 1e-9))
EER2 = (FAR2 + FRR2) / 2

classes2 = clf_fn.classes_
y_te_bin = label_binarize(y_te, classes=classes2)
auc2 = np.mean([
    auc(*roc_curve(y_te_bin[:, i], y_prob_fn[:, i])[:2])
    for i in range(len(classes2))
    if y_te_bin[:, i].sum() > 0
])

print("\n" + "="*40)
print("--- FACENET (DEEP LEARNING) SONUÇLARI ---")
print("="*40)
print(f"Accuracy (Doğruluk): %{acc_fn*100:.2f}")
print(f"FAR (Yanlış Kabul):  %{FAR2*100:.2f}")
print(f"FRR (Yanlış Red):    %{FRR2*100:.2f}")
print(f"EER (Eşit Hata):     %{EER2*100:.2f}")
print(f"AUC:                  {auc2:.4f}")
print("="*40)
print("Tamamlandı!")

26-04-28 12:27:54 - Directory /root/.deepface has been created
26-04-28 12:27:54 - Directory /root/.deepface/weights has been created
FaceNet embedding çıkarılıyor (2-3 dakika sürebilir)...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5
To: /root/.deepface/weights/facenet_weights.h5


26-04-28 12:27:59 - 🔗 facenet_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5 to /root/.deepface/weights/facenet_weights.h5...


100%|██████████| 92.2M/92.2M [00:00<00:00, 266MB/s]


  0/3024 tamamlandı...
  50/3024 tamamlandı...
  100/3024 tamamlandı...
  150/3024 tamamlandı...
  200/3024 tamamlandı...
  250/3024 tamamlandı...
  300/3024 tamamlandı...
  350/3024 tamamlandı...
  400/3024 tamamlandı...
  450/3024 tamamlandı...
  500/3024 tamamlandı...
  550/3024 tamamlandı...
  600/3024 tamamlandı...
  650/3024 tamamlandı...
  700/3024 tamamlandı...
  750/3024 tamamlandı...
  800/3024 tamamlandı...
  850/3024 tamamlandı...
  900/3024 tamamlandı...
  950/3024 tamamlandı...
  1000/3024 tamamlandı...
  1050/3024 tamamlandı...
  1100/3024 tamamlandı...
  1150/3024 tamamlandı...
  1200/3024 tamamlandı...
  1250/3024 tamamlandı...
  1300/3024 tamamlandı...
  1350/3024 tamamlandı...
  1400/3024 tamamlandı...
  1450/3024 tamamlandı...
  1500/3024 tamamlandı...
  1550/3024 tamamlandı...
  1600/3024 tamamlandı...
  1650/3024 tamamlandı...
  1700/3024 tamamlandı...
  1750/3024 tamamlandı...
  1800/3024 tamamlandı...
  1850/3024 tamamlandı...
  1900/3024 tamamlandı...
  1950/30

In [5]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc
import numpy as np

print("Hibrit model oluşturuluyor (PCA + FaceNet ensemble)...")

# Her iki modelin test setindeki ortak örnekleri bul
# PCA tarafı: y_test, y_prob (Hücre 2'den)
# FaceNet tarafı: y_te, y_prob_fn (Hücre 3'ten)

# Ortak sınıflar
pca_classes = list(clf.classes_)
fn_classes = list(clf_fn.classes_)
common_classes = sorted(set(pca_classes) & set(fn_classes))

# PCA olasılıklarını ortak sınıflara göre filtrele
pca_class_idx = [pca_classes.index(c) for c in common_classes]
fn_class_idx = [fn_classes.index(c) for c in common_classes]

# Test setlerini ortak sınıflara göre filtrele
pca_mask = np.isin(y_test, common_classes)
fn_mask = np.isin(y_te, common_classes)

y_test_common = y_test[pca_mask]
y_te_common = y_te[fn_mask]

pca_prob_common = y_prob[pca_mask][:, pca_class_idx]
fn_prob_common = y_prob_fn[fn_mask][:, fn_class_idx]

# İki modelin test setlerindeki ortak örnekleri eşleştir
# Stratified split aynı random_state kullandığı için oranlar benzer
# En küçük set kadar al
min_len = min(len(y_test_common), len(y_te_common))
y_hybrid_true = y_test_common[:min_len]
pca_p = pca_prob_common[:min_len]
fn_p = fn_prob_common[:min_len]

# Ağırlıklı ortalama: FaceNet daha güçlü olduğu için 0.7/0.3
hybrid_prob = 0.3 * pca_p + 0.7 * fn_p
y_hybrid_pred = np.array(common_classes)[np.argmax(hybrid_prob, axis=1)]

# Metrikler
acc_h = accuracy_score(y_hybrid_true, y_hybrid_pred)
cm_h = confusion_matrix(y_hybrid_true, y_hybrid_pred, labels=common_classes)
FP_h = cm_h.sum(axis=0) - np.diag(cm_h)
FN_h = cm_h.sum(axis=1) - np.diag(cm_h)
TP_h = np.diag(cm_h)
TN_h = cm_h.sum() - (FP_h + FN_h + TP_h)

FAR_h = np.mean(FP_h / (FP_h + TN_h + 1e-9))
FRR_h = np.mean(FN_h / (FN_h + TP_h + 1e-9))
EER_h = (FAR_h + FRR_h) / 2

y_true_bin = label_binarize(y_hybrid_true, classes=common_classes)
auc_h = np.mean([
    auc(*roc_curve(y_true_bin[:, i], hybrid_prob[:, i])[:2])
    for i in range(len(common_classes))
    if y_true_bin[:, i].sum() > 0
])

print("\n" + "="*40)
print("--- HİBRİT MODEL (PCA + FaceNet) SONUÇLARI ---")
print("="*40)
print(f"Accuracy (Doğruluk): %{acc_h*100:.2f}")
print(f"FAR (Yanlış Kabul):  %{FAR_h*100:.2f}")
print(f"FRR (Yanlış Red):    %{FRR_h*100:.2f}")
print(f"EER (Eşit Hata):     %{EER_h*100:.2f}")
print(f"AUC:                  {auc_h:.4f}")
print("="*40)
print("Tamamlandı!")

Hibrit model oluşturuluyor (PCA + FaceNet ensemble)...

--- HİBRİT MODEL (PCA + FaceNet) SONUÇLARI ---
Accuracy (Doğruluk): %74.55
FAR (Yanlış Kabul):  %0.26
FRR (Yanlış Red):    %25.43
EER (Eşit Hata):     %12.84
AUC:                  0.9862
Tamamlandı!
